# Tutorial: Differential Privacy - Noise and Accounting

**Prerequisites**: Tutorial 01 (Gradient Clipping), Basic understanding of privacy concepts

---

## Overview

In Tutorial 01, we learned **gradient clipping** - how to bound the sensitivity of per-example gradients. But clipping alone doesn't provide privacy! To achieve **differential privacy (DP)**, we need two more components:

1. ✅ **Gradient clipping** (Tutorial 01) - Bounds sensitivity
2. 🎯 **Noise injection** (This tutorial) - Adds calibrated Gaussian noise
3. 📊 **Privacy accounting** (This tutorial) - Tracks privacy budget (ε, δ)

This tutorial focuses on **understanding and using** noise and accounting - the theoretical foundations of DP-SGD. We'll save the complete training loop for Tutorial 03.

**Learning objectives**:
1. Understand why noise is essential for differential privacy
2. Use `gaussian_noise()` to add calibrated noise to gradients
3. Track privacy budget using the composable `DpProcess` API
4. Understand truncated Poisson sampling for stable batch sizes
5. Use calibration functions to find optimal hyperparameters for target privacy budgets

**What you'll build**:
- Manual noise addition to clipped gradients
- Privacy accounting for different sampling strategies
- Automated hyperparameter calibration for privacy budgets

---

## What is Differential Privacy?

**Informal definition**: An algorithm is differentially private if its output is nearly identical whether or not any single individual's data is included.

**Formal definition**: A randomized algorithm $\mathcal{M}$ satisfies $(\varepsilon, \delta)$-differential privacy if for all datasets $D$ and $D'$ differing in one row, and all outputs $S$:

$$P[\mathcal{M}(D) \in S] \leq e^\varepsilon \cdot P[\mathcal{M}(D') \in S] + \delta$$

**Key parameters**:
- **ε (epsilon)**: Privacy loss - smaller is better (typical: 1-10)
- **δ (delta)**: Failure probability - very small (typical: 1e-5 to 1e-7)

**Intuition**:
- ε ≈ 0: Perfect privacy (output independent of any individual)
- ε ≈ 1: Strong privacy
- ε ≈ 10: Weak privacy
- δ: Probability that privacy guarantee fails (should be << 1/dataset_size)

---

## Why Do We Need Noise?

**Gradient clipping alone is NOT private!**

Consider two datasets:
- Dataset A: [example_1, example_2, example_3]
- Dataset B: [example_1, example_2, example_4]  # different third example

If we clip gradients to norm 1.0:
```python
grad_A = clip(grad_1) + clip(grad_2) + clip(grad_3)
grad_B = clip(grad_1) + clip(grad_2) + clip(grad_4)
```

The outputs are **deterministic and different** - an adversary can tell which dataset was used!

**Solution**: Add Gaussian noise scaled to the sensitivity:
```python
grad_A_private = grad_A + N(0, σ²)  # σ = noise_multiplier × clip_norm
grad_B_private = grad_B + N(0, σ²)
```

Now the outputs are **random and similar** - harder to distinguish!

---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F

# Opaque imports
import opaque.accounting as acc
from opaque import make_functional, clipped_grad, gaussian_noise
from opaque import bounded_gaussian_noise

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")
print(f"Opaque imported successfully!")

---

## Part 1: Understanding Gaussian Noise

Let's start by understanding how Gaussian noise works.

In [3]:
# Create a simple gradient (simulating clipped and summed gradients)
grad = torch.tensor([1.0, 2.0, 3.0])

print(f"Original gradient: {grad}")
print(f"L2 norm: {grad.norm().item():.4f}")

Original gradient: tensor([1., 2., 3.])
L2 norm: 3.7417


In [ ]:
# Add Gaussian noise with stddev = 1.0
noise_fn, state = gaussian_noise(stddev=1.0)
noisy_grad, state = noise_fn(grad, state)

print(f"Noisy gradient: {noisy_grad}")
print(f"Noise added: {noisy_grad - grad}")
print(f"L2 norm of noise: {(noisy_grad - grad).norm().item():.4f}")

### Visualizing Noise Distribution

Let's see what happens when we add noise many times:

In [ ]:
# Generate many noisy versions
n_samples = 1000
original = torch.tensor([5.0])  # Single value for simplicity
stddev = 1.0

noisy_samples = []
for i in range(n_samples):
  noise_fn, state = gaussian_noise(stddev=stddev, generator=i)
  noisy, state = noise_fn(original, state)
  noisy_samples.append(noisy.item())

noisy_samples = np.array(noisy_samples)

# Plot histogram
plt.figure(figsize=(10, 5))
plt.hist(noisy_samples, bins=50, density=True, alpha=0.7, edgecolor='black')
plt.axvline(original.item(), color='red', linestyle='--', linewidth=2,
            label=f'True value = {original.item()}')
plt.xlabel('Value', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.title(f'Distribution of Noisy Gradient (stddev={stddev})', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean of noisy samples: {noisy_samples.mean():.4f} (should be ≈ {original.item()})")
print(f"Std of noisy samples:  {noisy_samples.std():.4f} (should be ≈ {stddev})")

**Key insight**:
- Noise is centered around the true value (unbiased)
- Higher stddev = more noise = better privacy, but worse utility
- This is the **privacy-utility tradeoff**

---

## Part 2: DP-SGD Algorithm

**Standard SGD**:
1. Compute gradient: $g = \frac{1}{B} \sum_{i=1}^B \nabla \ell(x_i, \theta)$
2. Update: $\theta \leftarrow \theta - \eta g$

**DP-SGD** (Differentially Private SGD):
1. Compute per-example gradients: $g_i = \nabla \ell(x_i, \theta)$
2. **Clip** each gradient: $\bar{g}_i = g_i / \max(1, \|g_i\|_2 / C)$
3. **Sum** clipped gradients: $\bar{g} = \sum_{i=1}^B \bar{g}_i$
4. **Add noise**: $\tilde{g} = \bar{g} + \mathcal{N}(0, \sigma^2 C^2 I)$
5. **Update**: $\theta \leftarrow \theta - \frac{\eta}{B} \tilde{g}$

Where:
- $C$ = clipping norm (sensitivity bound)
- $\sigma$ = noise multiplier (typically 0.5-2.0)
- $B$ = batch size
- $\eta$ = learning rate

**Important**:
- Noise stddev = `noise_multiplier × clip_norm`
- Noise is added to the **sum**, not individual gradients
- We still normalize by batch size (for learning rate stability)

---

## Part 3: Adding Noise to Clipped Gradients

Let's extend our gradient clipping example from Tutorial 01 with noise.

### Alternative: Bounded Gaussian Noise

If you need noisy values to stay within a specific range (e.g., valid gradient bounds), use
`bounded_gaussian_noise()`. It samples from a **truncated normal** distribution restricted to `[lower, upper]`
([Chen & Hale, 2024](https://arxiv.org/abs/2211.17230)).

In [ ]:
# Bounded Gaussian: outputs guaranteed within [-3, 3]
bounded_fn, state = bounded_gaussian_noise(stddev=1.0, bounds=(-3.0, 3.0))
noisy_bounded, state = bounded_fn(grad, state)

print(f"Original gradient:     {grad}")
print(f"Bounded noisy gradient: {noisy_bounded}")
print(f"All values in [-3, 3]: {(noisy_bounded >= -3.0).all() and (noisy_bounded <= 3.0).all()}")

# Compare: standard gaussian has no bounds
unbounded_fn, state2 = gaussian_noise(stddev=1.0)
noisy_unbounded, state2 = unbounded_fn(grad, state2)
print(f"\nUnbounded noisy gradient: {noisy_unbounded}")
print(f"(may exceed [-3, 3])")

In [6]:
# Setup: Model and data (same as Tutorial 01)
class SimpleMLP(nn.Module):
  def __init__(self, input_dim=10):
    super().__init__()
    self.fc1 = nn.Linear(input_dim, 64)
    self.fc2 = nn.Linear(64, 32)
    self.fc3 = nn.Linear(32, 1)

  def forward(self, x):
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    return self.fc3(x).squeeze(-1)


# Generate data
def generate_data(n_samples=10000, input_dim=10, seed=42):
  torch.manual_seed(seed)
  X = torch.randn(n_samples, input_dim)
  y = (X[:, 0] > 0).float()
  return X, y


X_train, y_train = generate_data()
print(f"Data: {X_train.shape}")

Data: torch.Size([200, 10])


In [7]:
# Convert to functional
model = SimpleMLP()
fmodel, params = make_functional(model)


# Define loss
def loss_fn(params, example):
  x, y = example
  logit = fmodel(params, x)
  return F.binary_cross_entropy_with_logits(logit, y)


# Create clipped gradient function (returns fn + state)
clip_norm = 1.0
clipped_grad_fn, clip_state = clipped_grad(
  loss_fn,
  argnums=0,
  batch_argnums=1,
  l2_clip_norm=clip_norm,
)

print(f"Clipping norm: {clip_norm}")
print(f"Sensitivity bound: {clip_state.sensitivity()}")

Clipping norm: 1.0
Sensitivity bound: 1.0


In [29]:
# Compute clipped gradients for a batch
X_batch = X_train[:32]
y_batch = y_train[:32]

clipped_grads, clip_state = clipped_grad_fn(params, (X_batch, y_batch), state=clip_state)

print("Clipped gradients (summed over batch):")
for i, g in enumerate(clipped_grads):
  print(f"  param[{i}]: shape {g.shape}, L2 norm {g.norm().item():.4f}")

Clipped gradients (summed over batch):
  param[0]: shape torch.Size([64, 10]), L2 norm 1.6634
  param[1]: shape torch.Size([64]), L2 norm 0.6243
  param[2]: shape torch.Size([32, 64]), L2 norm 5.4939
  param[3]: shape torch.Size([32]), L2 norm 2.0269
  param[4]: shape torch.Size([1, 32]), L2 norm 5.3438
  param[5]: shape torch.Size([1]), L2 norm 5.6306


### Add Gaussian Noise

Now let's add noise calibrated for differential privacy:

In [ ]:
# Noise parameters
noise_multiplier = 1.0
stddev = noise_multiplier * clip_norm  # stddev = noise_multiplier × sensitivity

print(f"Noise multiplier: {noise_multiplier}")
print(f"Clip norm (sensitivity): {clip_norm}")
print(f"Noise stddev: {stddev}")

# Add noise to clipped gradients
noise_fn, noise_state = gaussian_noise(stddev=stddev, generator=42)
noisy_grads, noise_state = noise_fn(clipped_grads, noise_state)

print("\nNoisy gradients:")
for i, (clean_g, noisy_g) in enumerate(zip(clipped_grads, noisy_grads)):
  noise_norm = (noisy_g - clean_g).norm().item()
  print(f"  param[{i}]: clean norm {clean_g.norm().item():.4f}, "
        f"noisy norm {noisy_g.norm().item():.4f}, "
        f"noise norm {noise_norm:.4f}")

### Understanding the Privacy-Utility Tradeoff

Let's see how different noise multipliers affect gradient quality:

In [ ]:
# Experiment: Add different amounts of noise to the same clipped gradient
clip_norm = 1.0
noise_multipliers = [0.5, 1.0, 2.0, 5.0]

# Use the clipped gradients from before
print(f"Original clipped gradient norm: {clipped_grads[0].norm().item():.4f}\n")

for nm in noise_multipliers:
  stddev = nm * clip_norm
  noise_fn, state = gaussian_noise(stddev=stddev, generator=42)
  noisy_grad, state = noise_fn(clipped_grads[0], state)

  noise_added = (noisy_grad - clipped_grads[0]).norm().item()
  signal_to_noise = clipped_grads[0].norm().item() / noise_added

  print(f"Noise multiplier = {nm:.1f}:")
  print(f"  Noise stddev: {stddev:.2f}")
  print(f"  Noise added (L2): {noise_added:.2f}")
  print(f"  Signal-to-noise ratio: {signal_to_noise:.2f}")
  print(f"  → {'Strong signal' if signal_to_noise > 1 else 'Weak signal (noise dominates)'}\n")

**Key observations**:
- Higher noise multiplier = more noise = worse gradient signal
- When signal-to-noise < 1, noise dominates (training will be difficult)
- When signal-to-noise > 1, signal is stronger (better training, less privacy)
- This is the **privacy-utility tradeoff**!

**Question**: How do we choose the right noise multiplier?
**Answer**: Privacy accounting + calibration (next sections)!

---

**Observations**:
- Noise magnitude is comparable to gradient magnitude
- This is necessary for privacy, but reduces gradient quality
- Higher noise_multiplier = better privacy, worse utility

---

## Part 4: Privacy Accounting

**Key question**: How much privacy budget (ε, δ) did we spend?

Privacy accounting tracks the cumulative privacy cost over multiple training steps.

### Composable Privacy Accounting API

Opaque uses a composable `DpProcess` API. Each mechanism returns a process,
`*` repeats it, and `|` composes different processes.

```python
# Create one DP-SGD step
step = acc.poisson(noise_multiplier=1.0, sample_rate=0.01)

# Compose 100 steps
training = step * 100

# Query privacy
epsilon = training.epsilon_at(1e-5)
```

**Benefits**:
- Explicit composition (no hidden state)
- Easy to branch and compare training plans
- Same API for all privacy metrics

### Sampling Mechanisms

1. **Poisson sampling** (`acc.poisson`):
   - Standard DP-SGD mechanism
   - Random batch sizes

2. **Truncated Poisson** (`acc.truncated_poisson`):
   - Batch size capped at a fixed maximum
   - Tighter privacy bounds in practice
   - Recommended for production DP-SGD

Let's see each in action!

In [31]:
# Identity process (zero privacy cost)
identity = acc.identity()

epsilon_initial = identity.epsilon_at(1e-5)
print("Initial process:")
print(f"  Epsilon (ε): {epsilon_initial:.4f}")
print("  Delta (δ): 1e-5")
print("  (This is the identity - no privacy spent yet)")

RDP Accountant created!
Initial epsilon: 0.0000


In [11]:
# One Poisson-sampled DP-SGD step
noise_multiplier = 1.1
sample_rate = 32 / 10000  # batch_size / dataset_size

step = acc.poisson(
  noise_multiplier=noise_multiplier,
  sample_rate=sample_rate,
)

epsilon_1 = step.epsilon_at(1e-5)
print("After 1 step:")
print(f"  Epsilon (ε): {epsilon_1:.4f}")
print("  Delta (δ): 1e-5")
print(f"  Interpretation: (ε={epsilon_1:.2f}, δ=1e-5)-DP")

After 1 step(s):
  Epsilon (ε): 6.6463
  Delta (δ): 1e-5
  Interpretation: (ε=6.65, δ=1e-5)-DP


In [32]:
# Compose 100 steps (repeat the same step)
training_100 = step * 100
epsilon_100 = training_100.epsilon_at(1e-5)

print("After 100 steps:")
print(f"  Epsilon (ε): {epsilon_100:.4f}")
print("  Delta (δ): 1e-5")
print(f"\n  Privacy degraded from {epsilon_1:.2f} to {epsilon_100:.2f} (as expected!)")

# Identity is unchanged (no hidden state)
print(f"\n  Identity still has ε = {identity.epsilon_at(1e-5):.4f}")

After 100 steps:
  Epsilon (ε): 0.5638
  Delta (δ): 0.5e-4

  Privacy degraded from 6.65 to 0.56 (as expected!)


**Key insight**: Privacy budget **increases** (gets worse) with more training steps!

### Comparing Sampling Methods

Let's compare standard Poisson vs fixed-size vs truncated Poisson:

In [33]:
# Same parameters for fair comparison
noise_mult = 1.1
sample_rate = 32 / 10000
num_steps = 100
batch_size = 32
dataset_size = 10000

# Method 1: Poisson sampling (random batch sizes)
step_poisson = acc.poisson(
  noise_multiplier=noise_mult,
  sample_rate=sample_rate,
)
training_poisson = step_poisson * num_steps

# Method 2: Truncated Poisson (bounded random batch sizes)
step_truncated = acc.truncated_poisson(
  noise_multiplier=noise_mult,
  sample_rate=sample_rate,
  batch_size_cap=batch_size,
  dataset_size=dataset_size,
)
training_truncated = step_truncated * num_steps

# Compare epsilons
eps_poisson = training_poisson.epsilon_at(1e-5)
eps_truncated = training_truncated.epsilon_at(1e-5)

print(f"After {num_steps} steps with noise_multiplier={noise_mult}:")
print(f"  Poisson:   ε = {eps_poisson:.4f}")
print(f"  Truncated: ε = {eps_truncated:.4f}")
print("\n  Truncated Poisson has tighter bounds in practice!")

After 100 steps:
  RDP: ε = 0.5638
  PLD: ε = 7.0284

  PLD is tighter by -6.4646 epsilon!
  (Lower epsilon = better privacy for same utility)


### Why Truncated Poisson is Better

Standard Poisson sampling has **variable batch sizes**:
- Can be 0 (rare but possible)
- Can be 2× or 3× the expected size
- Unpredictable memory usage
- Conservative worst-case privacy analysis

**Truncated Poisson** solves this:
- Batch size bounded to `truncated_batch_size`
- Still maintains randomness (DP requirement)
- Tighter privacy bounds = better utility
- Predictable resource usage

**Use truncated Poisson when**:
- You need stable batch sizes
- You want best privacy-utility tradeoff
- You're implementing production DP-SGD

In [ ]:
# Visualize privacy degradation over steps
steps_range = [1, 10, 50, 100, 200, 500]
epsilons_poisson = []
epsilons_truncated = []

for n_steps in steps_range:
  # Poisson
  training_p = acc.poisson(1.1, 0.0032) * n_steps
  epsilons_poisson.append(training_p.epsilon_at(1e-5))

  # Truncated
  training_t = acc.truncated_poisson(
    1.1, 0.0032, batch_size_cap=32, dataset_size=10000
  ) * n_steps
  epsilons_truncated.append(training_t.epsilon_at(1e-5))

plt.figure(figsize=(10, 6))
plt.plot(steps_range, epsilons_poisson, 'o-', label='Poisson', linewidth=2, markersize=8)
plt.plot(steps_range, epsilons_truncated, 's-', label='Truncated Poisson', linewidth=2, markersize=8)
plt.xlabel('Training Steps', fontsize=12)
plt.ylabel('Epsilon (ε)', fontsize=12)
plt.title('Privacy Degradation: Poisson vs Truncated Poisson', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Truncated Poisson consistently gives tighter bounds!")

---

## Part 5: Calibrating Noise

**Problem**: How do we choose `noise_multiplier` to achieve a target privacy budget?

**Solution**: Use calibration functions!

### Calibrate Noise Multiplier

Given:
- Target (ε, δ)
- Training parameters (sample_rate, num_steps)

Find:
- Minimum `noise_multiplier` that achieves target privacy

In [35]:
# Calibrate noise for target privacy
import opaque.accounting as acc

target_epsilon = 3.0
target_delta = 1e-5
sample_rate = 32 / 10000
num_steps = 1000

# Binary search for noise multiplier
target = acc.epsilon(target_epsilon, delta=target_delta)
result = acc.calibrate(
    target=target,
    build=lambda nm: acc.poisson(nm, sample_rate) * num_steps,
    param_min=0.1,
    param_max=5.0,
)

noise_mult = result.param
print(f"Target privacy: (ε={target_epsilon}, δ={target_delta})")
print(f"Training: sample_rate={sample_rate:.4f}, num_steps={num_steps}")
print(f"Calibrated noise_multiplier: {noise_mult:.4f}")
print(f"Achieved epsilon: {result.achieved:.6f}")

Target privacy: (ε=3.0, δ=1e-05)
Training: sample_rate=0.0032, num_steps=1000

Calibrated noise_multiplier: 0.9845


In [ ]:
# Verify calibration
training = acc.poisson(noise_mult, sample_rate) * num_steps
achieved_eps = training.epsilon_at(target_delta)

print(f"\nVerification:")
print(f"  Target ε:    {target_epsilon:.4f}")
print(f"  Achieved ε:  {achieved_eps:.4f}")
print(f"  Difference:  {abs(achieved_eps - target_epsilon):.6f}")
print(f"\n✓ Calibration successful!" if abs(achieved_eps - target_epsilon) < 0.1 else "\n✗ Calibration failed!")

### Calibration API Overview

All calibration functions follow this pattern:

```python
# Define target (what you want)
target = acc.epsilon(epsilon=3.0, delta=1e-5)

# Define build function (how you compose mechanisms)
build = lambda nm: acc.poisson(nm, sample_rate) * num_steps

# Binary search finds the parameter
result = acc.calibrate(target, build, param_min=0.1, param_max=5.0)

# Use the result
noise_multiplier = result.param
```

Available targets:
- `acc.epsilon(epsilon, delta)` - Standard (ε, δ)-DP
- `acc.delta(delta, epsilon)` - Inverse: find δ for given ε
- `acc.advantage(advantage)` - f-DP advantage
- `acc.beta(beta, alpha)` - Error rates
- `acc.risk(risk, prior)` - Bayes risk

In [ ]:
# Compare calibration across different sampling methods
target_eps = 3.0
target_delta = 1e-5
num_steps = 1000

target = acc.epsilon(target_eps, delta=target_delta)

# Method 1: Poisson (default)
result_poisson = acc.calibrate(
  target=target,
  build=lambda nm: acc.poisson(nm, 32/10000) * num_steps,
  param_min=0.1,
  param_max=5.0,
)

# Method 2: Truncated Poisson (recommended!)
result_truncated = acc.calibrate(
  target=target,
  build=lambda nm: acc.truncated_poisson(nm, 32/10000, batch_size_cap=32, dataset_size=10000) * num_steps,
  param_min=0.1,
  param_max=5.0,
)

print(f"Calibrated noise multipliers for ε={target_eps}, δ={target_delta}:")
print(f"  Poisson:           {result_poisson.param:.4f}")
print(f"  Truncated Poisson: {result_truncated.param:.4f}")
print(f"\nTruncated Poisson has better bounds (lower noise for same privacy)!")

In [ ]:
# Verify both methods achieve their targets
print(f"\nVerification (both should achieve ε ≈ {target_eps}):")

# Poisson
training_p = acc.poisson(result_poisson.param, 32/10000) * num_steps
eps_p = training_p.epsilon_at(target_delta)
print(f"  Poisson:           ε = {eps_p:.4f} (error: {abs(eps_p - target_eps):.4f})")

# Truncated
training_t = acc.truncated_poisson(
  result_truncated.param, 32/10000, batch_size_cap=32, dataset_size=10000
) * num_steps
eps_t = training_t.epsilon_at(target_delta)
print(f"  Truncated Poisson: ε = {eps_t:.4f} (error: {abs(eps_t - target_eps):.4f})")

print(f"\n✓ All methods successfully calibrated to target privacy!")

**Key insight**: All three metrics (epsilon, advantage, error rates) are different ways to express the same privacy guarantee. Use whichever makes most sense for your use case.

### Other Calibration Functions

Opaque provides calibration functions for different privacy metrics:

1. **`acc.epsilon()`** - (ε, δ)-DP (what we've been using)
2. **`acc.advantage()`** - f-DP advantage metric
3. **`acc.beta()`** - (α, β) error rates

All use the same `calibrate()` function with different target types.

In [37]:
# Example: Calibrate for error rates (α, β)
alpha = 1e-4  # False positive rate
target_beta = 0.8  # Target false negative rate

target_beta_obj = acc.beta(target_beta, alpha=alpha)
result_beta = acc.calibrate(
  target=target_beta_obj,
  build=lambda nm: acc.poisson(nm, 16/2048) * 1000,
  param_min=0.1,
  param_max=5.0,
)

print(f"Target: β ≤ {target_beta} at α = {alpha}")
print(f"Calibrated noise_multiplier: {result_beta.param:.4f}")

# Verify
training_beta = acc.poisson(result_beta.param, 16/2048) * 1000
achieved_beta = training_beta.beta_at(alpha)
print(f"\nVerification:")
print(f"  Target β:    {target_beta:.4f}")
print(f"  Achieved β:  {achieved_beta:.4f}")

Target: β ≤ 0.8 at α = 0.0001
Training: sample_rate=0.0078, num_steps=1000

Calibrated noise_multiplier: 0.6234

Verification:
  Target β: 0.8000
  Achieved β: 0.7996
  ✓ Success!


---

## Summary

### What We Learned

1. **Gaussian Noise Injection**
   - `gaussian_noise(stddev, generator=...)` returns `(noise_fn, state)`
   - `noise_fn(grads, state) -> (noisy_grads, new_state)`  
   - `bounded_gaussian_noise(stddev, bounds, generator=...)` - Bounded noise via truncation
   - Noise added to **summed clipped gradients**, not per-example
   - Unbiased: noise centered around true value

2. **Composable Privacy Accounting** (DpProcess API)
   - **Immutable composition**: `step = acc.poisson(nm, rate)` → `training = step * 1000`
   - **Heterogeneous**: `combined = a | b` merges two processes
   - **Query any metric**: `epsilon_at()`, `delta_at()`, `advantage()`, `beta_at()`, `risk_at()`
   - Privacy bounds from single PLD computation

3. **Two Sampling Methods**
   - `acc.poisson()` - Standard, random batch sizes
   - `acc.truncated_poisson()` - Bounded batches, tighter bounds (recommended!)

4. **Binary Search Calibration**
   - Define target: `acc.epsilon(3.0, delta=1e-5)`
   - Define builder: `lambda nm: acc.poisson(nm, sr) * steps`
   - Search: `acc.calibrate(target, builder, min, max)` → `result.param`
   - Works for all metrics: epsilon/delta, advantage, error rates

5. **Privacy Budget Tracking**
   - DP budget increases (epsilon gets worse) with more training
   - Composition is the key: each step adds privacy loss
   - Truncated Poisson gives tightest bounds in practice

### Key Takeaways

- **DP = Clipping + Noise + Accounting** - all three are essential
- **Composable & Immutable** - build training plans without hidden state
- **Calibrate, don't guess** - use binary search for target privacy
- **Truncated Poisson is best** - bounded batch sizes + optimal privacy
- **Multiple metrics available** - choose epsilon, advantage, or error rates

### Complete DP-SGD Workflow

```python
import opaque.accounting as acc
from opaque import clipped_grad, gaussian_noise

# 1. Calibrate noise
target = acc.epsilon(3.0, delta=1e-5)
result = acc.calibrate(
    target=target,
    build=lambda nm: acc.truncated_poisson(nm, 0.01, 32, 10000) * 1000,
    param_min=0.1, param_max=5.0,
)
noise_mult = result.param

# 2. Setup clipping (returns fn + state)
clipped_grad_fn, clip_state = clipped_grad(
    loss_fn, l2_clip_norm=1.0, argnums=0, batch_argnums=1
)

# 3. Setup noise
noise_fn, noise_state = gaussian_noise(
    stddev=noise_mult * clip_state.sensitivity(), generator=42
)

# 4. Training loop
for batch in dataloader:
    # Compute clipped gradients
    grads, clip_state = clipped_grad_fn(params, batch, state=clip_state)
    
    # Add noise
    noisy_grads, noise_state = noise_fn(grads, noise_state)
    
    # Update model
    params = optimizer.step(params, noisy_grads)

# 5. Check final privacy
training = acc.truncated_poisson(noise_mult, 0.01, 32, 10000) * num_steps
epsilon = training.epsilon_at(1e-5)
print(f"Final privacy: (ε={epsilon:.2f}, δ=1e-5)")
```

---

## What's Next?

**Tutorial 03: Complete DP-SGD Training**
- Put clipping + noise + accounting together
- Compare DP vs non-private training
- Visualize privacy-utility tradeoff

**Tutorial 04: Optimizers & Adaptive Clipping**
- Use adaptive clipping to auto-tune clip norm
- Compare different TorchOpt optimizers

**Tutorial 05: Sampling & Microbatching**
- Privacy amplification through subsampling
- Memory-efficient training with gradient accumulation

---

## Exercises

1. **Noise exploration**: Experiment with different noise multipliers (0.5, 1.0, 2.0, 5.0) and visualize distributions

2. **Sampling comparison**: Compare Poisson vs Truncated Poisson privacy bounds for different sample rates and dataset sizes

3. **Calibration practice**:
   - Find noise for ε=1.0, δ=1e-5, 1000 steps, sr=0.001
   - Find noise for ε=0.5 (stronger privacy)
   - Find noise for ε=10.0 (weaker privacy)
   - What happens as ε gets smaller?

4. **Privacy budget tracking**: Plot how epsilon increases with training steps for different noise multipliers

5. **Multi-target calibration**: Calibrate for (epsilon, advantage, error rates) and compare the resulting noise multipliers

---

## Key Resources

- [Privacy Accounting Guide](../../user-guide/accounting.md)
- [Noise Addition Guide](../../user-guide/noise.md)
- [API Reference: Accounting](../../api/accounting.md)

---

**Questions?** Open an issue on [GitHub](https://github.com/evgri243/opaque/issues).

**Further Reading**:
- [Deep Learning with Differential Privacy](https://arxiv.org/abs/1607.00133) (Abadi et al., 2016)
- [The Bounded Gaussian Mechanism](https://arxiv.org/abs/2211.17230) (Chen & Hale, 2024)
- [Privacy Loss Distributions](https://arxiv.org/abs/2106.08567) (Koskela et al., 2021)